In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h_big = 5
w_big = 5

h_small = 3
w_small = 3

avg_len = 0.4

In [ ]:

h = 2
w = 0.5
avg_len = 0.15
shift = [0, 0]

y_shift = 1
shift = np.array([5/6, y_shift])
ipu, points, segment_edges, m, markers= periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift, angle = 0, two_dash = True)

finalMarkers = np.where(np.array(markers) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
m1 = m
finalMarkers1 = finalMarkers

In [ ]:

h = 2
w = 0.5
avg_len = 0.15
shift = [0, 0]

y_shift2 = 0
shift = np.array([5/6, y_shift2])
ipu, points, segment_edges, m, markers= periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift, angle = 0, two_dash = True)

finalMarkers = np.where(np.array(markers) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
m2 = m
finalMarkers2 = finalMarkers


In [ ]:
axis = 0

In [ ]:
m, markers = periodic_unit_helper.shift_and_merge_two_periodic_mesh(m1, finalMarkers1, m2, finalMarkers2, axis = 0)

In [ ]:
m, markers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, markers, axis = 1)



In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), markers)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [ipu.numVars() - 2], 1e-8
fixedVars, hessianShift = [], 1e-8

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 2


opts.niter = 500
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success

In [ ]:
name = 'merge_two_dash'
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx)
fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-8
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
cr = az_optimizer.optimize()

stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, hessianShift = 1e-10, fixedVars = [], filename = "{}/stiffness_merge_two_dash_{}_{}.png".format(result_folder, y_shift, y_shift2))


In [ ]:
points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_merge_two_dash_{}_{}.png".format(result_folder, y_shift, y_shift2))

In [ ]:

an = np.linspace(0, 2 * np.pi, 100)
fig, ax = plt.subplots(1, 1)
ax.plot(np.cos(an), np.sin(an))

ax.plot(points[:, 0], points[:, 1])

ax.set_aspect('equal', 'box')
ax.set_title('still a circle, auto-adjusted data limits', fontsize=10)

fig.tight_layout()

plt.show()

In [ ]:

render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/render_merge_two_dash_{}_{}.png".format(result_folder, y_shift, y_shift2))

np.save("{}/stiffness_values_merge_two_dash_{}_{}.npy".format(result_folder, y_shift, y_shift2), stiffness_values)
np.save("{}/sampled_alphas_merge_two_dash_{}_{}.npy".format(result_folder, y_shift, y_shift2), sampled_alphas)
np.save("{}/scale_factors_merge_two_dash_{}_{}.npy".format(result_folder, y_shift, y_shift2), get_deformation_scale_factors(ipu))

In [ ]:
# stiffness_values, sampled_alphas, 
get_deformation_scale_factors(ipu)

### Inflatable experiment

In [ ]:
# for i in range(2):
#     m, markers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, markers, axis = 0)
#     m, markers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, markers, axis = 0)

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), markers)

In [ ]:
fuse_boundary = True

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
isheet = inflation.InflatableSheet(m, fusedVtx = fusedVtx)

from tri_mesh_viewer import TriMeshViewer
sheet_viewer = TriMeshViewer(isheet, width=768, height=640)
sheet_viewer.showWireframe(True)

In [ ]:
sheet_viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion

fixedVars, hessianShift = [], 1e-8

isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.disableFusedRegionTensionFieldTheory(False)

isheet.pressure = 1


opts.niter = 500
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        sheet_viewer.update(scalarField=utils.getStrains(isheet)[:, 0])
cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)